# 📉 cryoDRGN — **convergence** analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ts387/cryodrgn/blob/claude/cryodrgn-colab-notebook-5mf3p3/cryoDRGN_colab_convergence.ipynb)

`cryodrgn_utils analyze_convergence` is the only cryoDRGN analysis that compares **epochs to
each other**. `analyze` and `analyze_landscape` each describe a single epoch; this one asks
whether the model has stopped changing, which is the question you actually need answered before
believing any of the others.

| # | Plot | What it measures |
|---|---|---|
| 00 | `total_loss` | training loss per epoch |
| 01 | `encoder_umaps` | latent UMAP at each sampled epoch, side by side |
| 02 | `encoder_latent_vector_shifts` | median magnitude, dot product and cosine distance of per-particle latent movement between consecutive epochs |
| 03 | `decoder_UMAP-sketching` | local maxima picked out of the UMAP density |
| 04 | `decoder_maxima-sketch-consistency` | whether particles stay near the same maximum |
| 05 | `decoder_CC` | map–map correlation between volumes decoded at the same latent point in different epochs |
| 06–07 | `decoder_FSC`, `decoder_FSC-nyquist` | the same comparison, resolution-resolved |

Plots 00–04 need no GPU. 05–07 regenerate volumes and want one.

### Read 02 and 05–07 first

Loss keeps drifting down long after the model has stopped changing in any way you care about.
The informative signals are the **latent shifts** (has each particle's embedding stopped
moving?) and the **map–map CC/FSC** (does the decoder produce the same density at the same
latent point two epochs apart?). Flat, high CC across the last few sampled epochs is what
convergence actually looks like.

### It is BETA, and it has three real bugs

The module is marked BETA in cryoDRGN 4.3.0. Three defects will stop it outright on a normal
`train_vae` run; this notebook patches all three and says so when it does. Details in the notes
at the bottom — briefly:

1. it asserts `z.0.pkl` exists, but `train_vae` numbers epochs from 1;
2. it makes masks from `vol_000.mrc` while `eval_vol` writes `vol_001.mrc`;
3. its UMAP montage indexes `epochs[i]` on the line *above* its own `try:`, so the
   `except IndexError` never fires and most epoch counts crash.

### Nothing is written to your model folder

All work happens in a local mirror, so the seeded `z.0.pkl` and everything else stays out of
your Drive run directory. Section 5 copies the results back.

## 1 · Setup

In [ ]:
#@title 1.1 · Check the GPU runtime { display-mode: "form" }
#@markdown Confirms a CUDA GPU is attached. If this prints **"No GPU found"**, go to
#@markdown **Runtime → Change runtime type → GPU** and re-run this cell.
import subprocess, sys

print("=" * 60)
gpu = subprocess.run(["nvidia-smi",
                      "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode == 0 and gpu.stdout.strip():
    name, mem, driver = [x.strip() for x in gpu.stdout.strip().split(",")]
    print(f"✅ GPU detected : {name}")
    print(f"   Memory       : {mem}")
    print(f"   Driver       : {driver}")
else:
    print("❌ No GPU found!")
    print("   Runtime → Change runtime type → Hardware accelerator → GPU,")
    print("   then re-run this cell. cryoDRGN training needs a GPU.")
print("=" * 60)

In [ ]:
#@title 1.2 · Install cryoDRGN { display-mode: "form" }
#@markdown Installs cryoDRGN from PyPI. Colab's pre-installed PyTorch/CUDA are kept.
#@markdown <br>• **stable** – the recommended release &nbsp;•&nbsp; **beta** – newest dev build from TestPyPI
release_channel = "stable"  #@param ["stable", "beta"]
#@markdown Optionally pin an exact version (e.g. `4.3.0`); leave blank for the latest.
version = ""  #@param {type:"string"}
#@markdown A few dependencies are pinned to versions other than Colab's defaults, so the
#@markdown runtime **restarts automatically** at the end. That is expected — just carry
#@markdown on with the next cell afterwards.
restart_after_install = True  #@param {type:"boolean"}

import subprocess, sys

pkg = "cryodrgn"
if version.strip():
    pkg = f"cryodrgn=={version.strip()}"

if release_channel == "beta":
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "-i", "https://test.pypi.org/simple/",
           "--extra-index-url", "https://pypi.org/simple/",
           "cryodrgn", "--pre"]
    if version.strip():
        cmd[cmd.index("cryodrgn")] = pkg
else:
    cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]

print("Installing", pkg, f"({release_channel} channel) — this takes ~1-2 min...\n")
ret = subprocess.run(cmd)
if ret.returncode != 0:
    raise SystemExit("❌ pip install failed — see the log above.")

# --- realign torchvision with torch -------------------------------------------------
# cryoDRGN pins torch<2.10, so pip may DOWNGRADE Colab's torch. Colab's pre-installed
# torchvision was compiled against the newer torch, and once they disagree importing it
# raises "operator torchvision::nms does not exist". That breaks EVERY cryodrgn command,
# because the CLI eagerly imports all command modules and analyze_landscape_full imports
# umap -> torchvision. Matching pair is torch 2.N <-> torchvision 0.(N+15).
import importlib.metadata as md

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

tver, tvver = _ver("torch"), _ver("torchvision")
if tver and tvver:
    tmaj, tmin = (int(x) for x in tver.split(".")[:2])
    tvmin = int(tvver.split(".")[1])
    want = tmin + 15 if tmaj == 2 else None
    if want is not None and tvmin != want:
        print(f"\n⚠️  torch {tver} and torchvision {tvver} are incompatible "
              f"(cryoDRGN's torch<2.10 pin downgraded torch).")
        print(f"   Installing torchvision 0.{want}.* to match...")
        fix = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
             f"torchvision==0.{want}.*"])
        if fix.returncode == 0:
            print(f"   ✅ torchvision realigned to 0.{want}.*")
        else:
            print(f"   ❌ Could not install torchvision 0.{want}.* — if cryodrgn commands "
                  f"fail with 'torchvision::nms does not exist', run:")
            print(f"      !pip install --no-deps 'torchvision==0.{want}.*'")

print("\n✅ cryoDRGN installed.")
if restart_after_install:
    print("🔄 Restarting the runtime to finalize the install (this is normal)...")
    print("   When it reconnects, continue from cell 1.3 — do NOT re-run this cell.")
    get_ipython().kernel.do_shutdown(True)

In [ ]:
#@title 1.3 · Verify the installation { display-mode: "form" }
#@markdown Run this **after** the runtime has restarted. The `cryodrgn --version` smoke-test is
#@markdown the important one: the CLI imports *every* command module on startup, so a broken
#@markdown dependency anywhere makes all commands fail — better to catch it here than mid-run.
import sys, subprocess
import torch, cryodrgn

print(f"cryoDRGN version : {cryodrgn.__version__}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device      : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  CUDA not available — fine for Step 4 (CPU-only), but Step 6 needs a GPU (cell 1.1).")

# torchvision must match torch or `import umap` blows up inside the cryodrgn CLI
try:
    import torchvision
    print(f"torchvision      : {torchvision.__version__} (ok)")
except Exception as e:
    msg = str(e).splitlines()[0]
    want = "0.%d.*" % (int(torch.__version__.split(".")[1]) + 15)
    if "numpy.dtype size changed" in msg or "binary incompatibility" in msg:
        # cryoDRGN pins numpy<1.27, downgrading Colab's numpy 2.x. C extensions that were
        # compiled against numpy 2.x headers then fail their ABI check on import.
        print(f"⚠️  torchvision fails a NumPy ABI check: {msg}")
        print("   Cause: cryoDRGN pins numpy<1.27, so Colab's numpy 2.x was downgraded and")
        print("   torchvision (built against numpy 2.x) no longer matches.")
        print("   This is USUALLY HARMLESS: cryoDRGN never imports torchvision itself — only")
        print("   umap does, and cryodrgn.analysis imports umap lazily. Every cryodrgn command")
        print("   also runs as a subprocess. Treat the CLI check below as the real verdict, and")
        print("   don't 'fix' this unless something actually fails.")
    else:
        print(f"❌ torchvision is broken: {msg}")
        print("   Looks like a torch/torchvision version mismatch rather than a NumPy issue.")
        print(f"   Fix with:  !pip install --no-deps 'torchvision=={want}'")
        print("   then re-run this cell (no restart needed).")

print("\n$ cryodrgn --version")
r = subprocess.run(["cryodrgn", "--version"], capture_output=True, text=True)
print((r.stdout + r.stderr).strip()[-2000:])
if r.returncode != 0:
    raise RuntimeError("The cryodrgn CLI failed to start — fix the error above before continuing.")
print("\n✅ CLI healthy — all command modules import cleanly.")

In [ ]:
#@title 1.4 · Mount Google Drive { display-mode: "form" }
#@markdown Click the link that appears, pick your Google account, and paste the code
#@markdown (or approve the pop-up). Your Drive appears under `/content/drive/MyDrive`.
from google.colab import drive
drive.mount("/content/drive")
print("\n✅ Drive mounted at /content/drive/MyDrive")

## 2 · Attach and preflight

`analyze_convergence` samples epochs as `np.arange(4, E+1, interval)`, appending `E` if it is not
already included (`analyze_convergence.py:1037-1040`). So it always starts at **epoch 4**, and
the interval decides how many points every downstream plot gets.

Cell 2.1 computes that list, checks the run actually has the files, and — because of bug 3 —
tells you whether the resulting count will survive the montage, offering intervals that do.

In [ ]:
#@title 2.1 · Point at the run and choose the epoch sampling { display-mode: "form" }
#@markdown Drive project folder.
drive_project_dir = "/content/drive/MyDrive/cryodrgn_project"  #@param {type:"string"}
#@markdown Model folder name inside it, e.g. `00_cryodrgn128`.
run_name = ""  #@param {type:"string"}
#@markdown Latest epoch to analyze — **-1** uses the newest `z.N.pkl`.
epoch = -1  #@param {type:"integer"}
#@markdown Epochs between the expensive checks. Sampling is `arange(4, E+1, interval)` + `E`.
epoch_interval = 5  #@param {type:"integer"}

import os, re, glob, math
import numpy as np

if not run_name.strip():
    raise ValueError("Set run_name to the model folder you want to analyze.")
DRIVE_DIR = os.path.abspath(drive_project_dir)
RUN = os.path.join(DRIVE_DIR, run_name.strip())
if not os.path.isdir(RUN):
    raise FileNotFoundError(f"{RUN} not found — check drive_project_dir / run_name.")
if not os.path.exists(os.path.join(RUN, "config.yaml")):
    raise FileNotFoundError(f"No config.yaml in {RUN} — is this a cryoDRGN run folder?")

have = sorted(int(m.group(1)) for p in glob.glob(os.path.join(RUN, "z.*.pkl"))
              for m in [re.search(r"z\.(\d+)\.pkl$", os.path.basename(p))]
              if m and int(m.group(1)) > 0)
if not have:
    raise FileNotFoundError(f"No numbered z.N.pkl in {RUN}.")
E = int(epoch) if int(epoch) > 0 else max(have)
if E not in have:
    raise FileNotFoundError(f"No z.{E}.pkl in {RUN}. Available: {have}")
if E < 5:
    raise ValueError(
        f"Epoch {E} is too early: sampling starts at epoch 4 and needs at least two points, "
        f"so E must be 5 or more. Train further first.")

# analyze_convergence asserts z.0 .. z.{E-1} all exist (analyze_convergence.py:1049-1053).
# z.0 does not exist on a 1-based run and is seeded in 2.2; the rest must genuinely be there.
missing = [e for e in range(1, E + 1) if e not in have]
if missing:
    raise FileNotFoundError(
        f"Convergence needs EVERY epoch from 1 to {E}: missing z.N.pkl for {missing}.\n"
        f"  It reads consecutive epochs to measure how far each particle moved, so a gap is\n"
        f"  fatal. Pick an epoch below the first gap: try epoch = {min(missing) - 1}.")

iv = int(epoch_interval)
if iv < 1:
    raise ValueError("epoch_interval must be >= 1.")
def sampled(interval):
    ep = np.arange(4, E + 1, interval)
    if len(ep) == 0:
        return ep
    return ep if ep[-1] == E else np.append(ep, E)
EPOCHS = sampled(iv)
if len(EPOCHS) < 2:
    raise ValueError(f"Only {len(EPOCHS)} sampled epoch(s) at interval {iv} — need >= 2. "
                     f"Reduce epoch_interval.")

# bug 3: the montage grid is ceil(sqrt(n)) x ceil(n/ceil(sqrt(n))); when that has more cells
# than epochs, epochs[i] raises on the spare axes -- outside the try that would catch it.
def grid_fits(n):
    c = math.ceil(n ** 0.5)
    return c * math.ceil(n / c) == n
print(f"run            : {RUN}")
print(f"epochs present : 1..{max(have)}")
print(f"analyzing to   : epoch {E}")
print(f"sampled epochs : {list(map(int, EPOCHS))}  ({len(EPOCHS)} points, interval {iv})")
if grid_fits(len(EPOCHS)):
    print("montage        : ✓ this count exactly fills its subplot grid")
else:
    alt = sorted({int(i) for i in range(1, max(2, E)) if len(sampled(i)) >= 2
                  and grid_fits(len(sampled(i)))})
    print(f"montage        : ⚠️  {len(EPOCHS)} epochs does NOT fill its grid — cryoDRGN would")
    print("                 crash here (bug 3). Cell 3.1 patches it and redraws, so this is")
    print("                 safe to ignore.")
    if alt:
        print(f"                 Intervals that avoid it entirely at E={E}: {alt[:8]}")
if len(EPOCHS) < 3:
    print("               : only 2 sampled epochs — every trend plot will have one segment. "
          "Lower epoch_interval for a readable curve.")

for k, v in dict(CV_DRIVE=DRIVE_DIR, CV_RUN=RUN, CV_NAME=run_name.strip(),
                 CV_E=str(E), CV_IV=str(iv),
                 CV_EPOCHS=",".join(str(int(x)) for x in EPOCHS)).items():
    os.environ[k] = v

In [ ]:
#@title 2.2 · Build the mirror workdir and resolve Å/px { display-mode: "form" }
#@markdown Whether to stage `weights.N.pkl` for the sampled epochs — needed only for the
#@markdown CC/FSC metrics (`full` mode in 3.1). They are the big files.
stage_weights = True  #@param {type:"boolean"}
#@markdown Pixel size for the generated volumes. **`0` resolves it from `ctf.pkl`.**
#@markdown `analyze_convergence` hardcodes a default of 1.0 and never reads the CTF.
apix = 0  #@param {type:"number"}
#@markdown Local scratch root.
work_root = "/content/convergence_work"  #@param {type:"string"}

import os, shutil
import numpy as np
import yaml
from cryodrgn import utils

RUN, E = os.environ["CV_RUN"], int(os.environ["CV_E"])
EPOCHS = [int(x) for x in os.environ["CV_EPOCHS"].split(",")]
cfg = yaml.safe_load(open(os.path.join(RUN, "config.yaml")))
D_box = cfg["lattice_args"]["D"] - 1

WORK = os.path.abspath(os.path.join(work_root, f"{os.environ['CV_NAME']}.{E}"))
os.makedirs(WORK, exist_ok=True)
shutil.copyfile(os.path.join(RUN, "config.yaml"), os.path.join(WORK, "config.yaml"))

# --- run.log ---------------------------------------------------------------------------
# main() asserts it exists (analyze_convergence.py:1047) and plot_loss parses it. train_vae
# holds it open for the whole run and Drive only uploads a file on close, so a run whose
# session died may not have one here. Stub it rather than fail: only plot 00 depends on it.
_srclog, _dstlog = os.path.join(RUN, "run.log"), os.path.join(WORK, "run.log")
if os.path.exists(_srclog):
    shutil.copyfile(_srclog, _dstlog)
elif not os.path.exists(_dstlog):
    open(_dstlog, "w").close()
    print("run.log   : absent from the run folder — using an empty one. Plot 00 (total loss)")
    print("            will be blank; every other metric is computed from the checkpoints.")

# --- latents: EVERY epoch 1..E, plus the seeded z.0 ------------------------------------
staged = 0
for e in range(1, E + 1):
    s, d = os.path.join(RUN, f"z.{e}.pkl"), os.path.join(WORK, f"z.{e}.pkl")
    if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(s):
        shutil.copyfile(s, d); staged += 1
z0 = os.path.join(WORK, "z.0.pkl")
if not os.path.exists(z0):
    shutil.copyfile(os.path.join(WORK, "z.1.pkl"), z0)
print(f"latents   : {E} epoch(s) staged ({staged} copied this run) + z.0.pkl seeded from z.1")
print("            (bug 1 — the seed makes the epoch-2 point of plot 02 degenerate; read that")
print("             plot from epoch 3 on. It lives in the mirror, never in your run folder.)")

# --- weights for the sampled epochs (full mode only) -----------------------------------
if stage_weights:
    tot = 0
    for e in EPOCHS:
        s, d = os.path.join(RUN, f"weights.{e}.pkl"), os.path.join(WORK, f"weights.{e}.pkl")
        if not os.path.exists(s):
            raise FileNotFoundError(
                f"weights.{e}.pkl is missing from {RUN}, but epoch {e} is in the sampled set "
                f"{EPOCHS} — CC/FSC cannot run. Untick stage_weights for a 'fast' analysis, "
                f"or change epoch_interval in 2.1 to avoid that epoch.")
        if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(s):
            print(f"staging   : weights.{e}.pkl ({os.path.getsize(s)/2**20:.0f} MB)", flush=True)
            shutil.copyfile(s, d)
        tot += os.path.getsize(d)
    print(f"weights   : {len(EPOCHS)} epoch(s), {tot/2**30:.2f} GiB — enough for 'full' mode")
else:
    print("weights   : not staged — 3.1 can only run 'fast' (plots 00-04)")

# --- Å/px -------------------------------------------------------------------------------
# Only reaches the MRC headers of the generated volumes: the CC is scale-free and the FSC
# x-axis is in 1/px. But a wrong header makes the maps unusable in ChimeraX, and --dilate
# /--dist here are in VOXELS (unlike analyze_landscape's Angstroms), so nothing else shifts.
APIX = float(apix)
if APIX <= 0:
    _ctf = cfg["dataset_args"].get("ctf")
    if _ctf and os.path.exists(_ctf):
        cp = np.asarray(utils.load_pkl(_ctf))
        aps, szs = set(cp[:, 1]), set(cp[:, 0])
        if len(aps) == 1:
            _ap, _sz = float(tuple(aps)[0]), float(tuple(szs)[0])
            APIX = round(_ap * _sz / D_box, 6)
            print(f"A/px      : {APIX:g} (from {os.path.basename(_ctf)}: "
                  f"{_ap:g} A/px at box {_sz:g} -> box {D_box})")
        else:
            APIX = 1.0
            print("A/px      : ctf.pkl has multiple optics groups — falling back to 1.0, as "
                  "cryodrgn analyze would. Set apix above to override.")
    else:
        APIX = 1.0
        print("A/px      : no readable ctf.pkl in config.yaml — falling back to 1.0. "
              "Set apix above if you want correct volume headers.")
else:
    print(f"A/px      : {APIX:g} (set explicitly)")

n_ptcl, zdim = np.asarray(utils.load_pkl(os.path.join(WORK, f"z.{E}.pkl"))).shape
print(f"particles : {n_ptcl:,}   zdim {zdim}   box {D_box}")
print(f"workdir   : {WORK}   ({shutil.disk_usage(WORK).free/2**30:.0f} GiB free)")
os.environ.update(CV_WORK=WORK, CV_APIX=str(APIX), CV_D=str(D_box),
                  CV_N=str(n_ptcl), CV_WEIGHTS="1" if stage_weights else "0")

## 3 · Run it

`fast` runs metrics 1–3 (plots 00–04) by calling the library functions directly. `full` calls
`main()`, which adds volume generation, masking and the CC/FSC comparisons.

`full` is not `--skip-volgen`-able in reverse: masking sits inside the same `else` branch as
volume generation, so `--skip-volgen` leaves `calculate_CCs` with no `.masked.mrc` to read and it
dies after writing plots 00–04. That is why `fast` calls the functions itself rather than passing
a flag.

Cost: `full` decodes `final_maxima × len(sampled epochs)` volumes. At the defaults for a
25-epoch run that is 10 × 6 = 60 volumes — minutes on a GPU, hours on CPU. `vol_downsample`
cuts it by the cube of the ratio.

In [ ]:
#@title 3.1 · analyze_convergence (patching its three bugs) { display-mode: "form" }
#@markdown `fast` = plots 00–04, CPU-friendly. `full` = adds CC/FSC, wants a GPU.
#@markdown `auto` picks by GPU presence.
mode = "auto"  #@param ["auto", "fast", "full"]
#@markdown Particles subsampled for the UMAPs. The full stack is far slower for no extra insight.
umap_subset = 50000  #@param {type:"integer"}
#@markdown `full` only: volumes per epoch (local maxima of the UMAP density).
final_maxima = 10  #@param {type:"integer"}
#@markdown `full` only: box size for those volumes (`0` = no downsampling).
vol_downsample = 0  #@param {type:"integer"}
#@markdown Reuse UMAPs cached by a previous run of this cell — the slowest CPU step.
reuse_umaps = True  #@param {type:"boolean"}

import os, glob, shutil, logging, argparse
import numpy as np
import matplotlib.pyplot as plt
import torch

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
from cryodrgn.commands_utils import analyze_convergence as ac
import cryodrgn.analysis as _analysis

WORK, E = os.environ["CV_WORK"], int(os.environ["CV_E"])
EPOCHS = np.array([int(x) for x in os.environ["CV_EPOCHS"].split(",")])
APIX, n_total = float(os.environ["CV_APIX"]), int(os.environ["CV_N"])
if int(final_maxima) < 3:
    raise ValueError(f"final_maxima must be >= 3 (got {final_maxima}).")

# ---- bug 2: masking reads vol_000.mrc, eval_vol writes vol_001.mrc -------------------
# mask_volumes iterates `for cluster in range(len(labels))` and opens vol_{cluster:03d}.mrc
# (analyze_convergence.py:830-833), i.e. 0-based; analysis.gen_volumes defaults to
# vol_start_index=1. Force it to 0 so the two agree.
if not getattr(_analysis.gen_volumes, "_zero_indexed", False):
    _gv = _analysis.gen_volumes
    def _gen_volumes_0(*a, **k):
        return _gv(*a, **{**k, "vol_start_index": 0})
    _gen_volumes_0._zero_indexed = True
    _analysis.gen_volumes = _gen_volumes_0
    ac.analysis.gen_volumes = _gen_volumes_0
    print("  patched: gen_volumes forced to vol_start_index=0 so mask_volumes finds its inputs")

# ---- bug 3: the UMAP montage's ragged subplot grid -----------------------------------
# encoder_latent_umaps builds a ceil(sqrt(N)) x ceil(N/ceil(sqrt(N))) grid then reads
# epochs[i] on the line ABOVE its own `try:` (analyze_convergence.py:306-307), so the
# `except IndexError` never fires and any N that does not exactly fill the grid dies on the
# spare axes. Every umap.N.pkl is written before the crash, so recover and redraw.
def _draw_umap_montage(cdir_, epochs_):
    n = len(epochs_)
    n_cols = int(np.ceil(n ** 0.5))
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2 * n_cols, 2 * n_rows),
                             sharex="all", sharey="all", squeeze=False)
    toplot = None
    for k, ax in enumerate(axes.flat):
        fl = os.path.join(cdir_, "umaps", f"umap.{epochs_[k]}.pkl") if k < n else None
        if fl is None or not os.path.exists(fl):
            ax.set_visible(False)
            continue
        emb = ac.utils.load_pkl(fl)
        toplot = ax.hexbin(emb[:, 0], emb[:, 1], bins="log", mincnt=1)
        ax.set_title(f"epoch {epochs_[k]}")
    for a in axes[:, 0]:
        a.set_ylabel("UMAP2")
    for a in axes[-1, :]:
        a.set_xlabel("UMAP1")
    fig.subplots_adjust(right=0.96)
    if toplot is not None:
        cbar = fig.colorbar(toplot, cax=fig.add_axes([0.98, 0.15, 0.02, 0.7]))
        cbar.ax.set_ylabel("particle density", rotation=90)
    plt.subplots_adjust(wspace=0.1, hspace=0.3)
    out_fl = os.path.join(cdir_, "plots", "01_encoder_umaps.png")
    plt.savefig(out_fl, dpi=300, format="png", transparent=True, bbox_inches="tight")
    plt.close("all")
    print(f"  redrew {out_fl}")

if not getattr(ac.encoder_latent_umaps, "_grid_safe", False):
    _elu = ac.encoder_latent_umaps
    def _encoder_latent_umaps_safe(*a, **k):
        try:
            _elu(*a, **k)
        except IndexError:
            plt.close("all")
            print("  recovered: encoder_latent_umaps' ragged-grid IndexError — every UMAP was")
            print("             already computed and saved, redrawing the montage")
            _draw_umap_montage(a[1], a[2])     # (workdir, outdir, epochs, ...) all positional
    _encoder_latent_umaps_safe._grid_safe = True
    ac.encoder_latent_umaps = _encoder_latent_umaps_safe
    print("  patched: encoder_latent_umaps guarded against its ragged-subplot-grid IndexError")

# ---- fast or full -------------------------------------------------------------------
has_gpu = torch.cuda.is_available()
run_mode = mode if mode != "auto" else ("full" if has_gpu else "fast")
if run_mode == "full" and os.environ.get("CV_WEIGHTS") != "1":
    raise RuntimeError(
        "mode='full' needs weights.N.pkl staged — re-run cell 2.2 with stage_weights ticked.")
print(f"  GPU visible: {has_gpu} -> mode = {run_mode}")
if run_mode == "full" and not has_gpu:
    print(f"  ⚠️  'full' on CPU decodes {int(final_maxima) * len(EPOCHS)} volumes — expect "
          f"HOURS. Set vol_downsample=64, or use mode='fast'.")

CDIR = os.path.join(WORK, f"convergence.{E}")
for sub in ("plots", "umaps", "repr_particles"):
    os.makedirs(os.path.join(CDIR, sub), exist_ok=True)

need = [os.path.join(CDIR, "umaps", f"umap.{e}.pkl") for e in EPOCHS]
need.append(os.path.join(CDIR, "ind_subset.pkl"))    # follow_candidate_particles needs it
skip_umap = bool(reuse_umaps) and all(os.path.exists(p) for p in need)
if reuse_umaps and not skip_umap:
    print("  reuse_umaps requested but the cache is incomplete — recomputing UMAPs.")
elif skip_umap:
    print(f"  reusing {len(EPOCHS)} cached UMAP(s) from a previous run")

if run_mode == "full":
    p = argparse.ArgumentParser()
    ac.add_args(p)
    argv = [WORK, str(E), "--outdir", CDIR,
            "--epoch-interval", os.environ["CV_IV"],
            "--subset", str(int(umap_subset)),
            "--random-seed", "0",             # else the UMAP subset differs run to run
            "--Apix", str(APIX),
            "--final-maxima", str(int(final_maxima))]
    if int(vol_downsample) > 0:
        argv += ["-d", str(int(vol_downsample))]
    if skip_umap:
        argv += ["--skip-umap"]
    print("$ cryodrgn_utils analyze_convergence " + " ".join(argv), "\n" + "=" * 70)
    plt.close("all")
    ac.main(p.parse_args(argv))
    plt.close("all")
else:
    print("=" * 70)
    logfile = os.path.join(WORK, "run.log")
    print("Convergence 1: total loss curve ...")
    plt.close("all"); plt.figure(figsize=(4, 3))
    ac.plot_loss(logfile, CDIR, E)
    plt.close("all")

    if skip_umap:
        print("Convergence 2: reusing cached UMAPs ...")
    else:
        print(f"Convergence 2: UMAP embeddings for epochs {list(map(int, EPOCHS))} "
              f"({min(n_total, int(umap_subset)):,} particles each) ...")
        use_gpu_umap = hasattr(ac, "cuUMAP")
        print("  backend:", "cuML (GPU)" if use_gpu_umap else "umap-learn (CPU)")
        ac.encoder_latent_umaps(WORK, CDIR, EPOCHS, n_total, int(umap_subset),
                                0, use_gpu_umap, 42, 25000)

    print(f"Convergence 3: latent encoding shifts over epochs 2-{E} ...")
    plt.close("all")
    with np.errstate(invalid="ignore", divide="ignore"):   # seeded z.0 makes point 1 NaN
        ac.encoder_latent_shifts(WORK, CDIR, E)
    plt.close("all")
    print("Skipping Convergence 4 (volume CC + FSC) — set mode='full' on a GPU runtime.")

os.environ["CV_CDIR"] = CDIR
print("=" * 70 + f"\n✅ results → {CDIR}\n   View with 4.1, copy to Drive with 5.1.")

## 4 · View

How to read them:

* **02 latent vector shifts.** Magnitude falling toward a floor means embeddings have stopped
  moving. Cosine distance rising toward **1** means consecutive epochs push particles in
  uncorrelated directions — jitter, not progress. Both are the sign you want. Ignore the
  epoch-2 point: it comes from the seeded `z.0.pkl`.
* **05 CC / 06–07 FSC.** Volumes decoded at the same latent point in consecutive sampled
  epochs. Curves that are high and flat across the last few epochs mean the decoder has
  settled. A curve still climbing means keep training.
* **01 UMAP montage.** The shape should stabilise. Rotations and reflections between epochs are
  meaningless — UMAP is not orientation-stable — so judge topology, not pose.
* **04 sketch consistency.** Whether the particles near each density maximum stay the same set.

In [ ]:
#@title 4.1 · Show the plots { display-mode: "form" }
width = 720  #@param {type:"integer"}

import os, glob
from IPython.display import Image, display, Markdown

CDIR = os.environ["CV_CDIR"]
CAPS = {
    "00_total_loss.png": "**00 · Total loss** — from `run.log`. Keeps drifting long after the "
                         "model has settled; the least informative plot here.",
    "01_encoder_umaps.png": "**01 · Latent UMAP per epoch.** Judge topology, not orientation — "
                            "UMAP rotates and reflects freely between runs.",
    "02_encoder_latent_vector_shifts.png":
        "**02 · Latent vector shifts.** Magnitude → a floor and cosine distance → 1 together "
        "mean the embeddings have stopped moving. **Ignore the first point** (seeded `z.0`).",
    "03_decoder_UMAP-sketching.png": "**03 · UMAP local maxima** chosen for volume comparison.",
    "04_decoder_maxima-sketch-consistency.png":
        "**04 · Sketch consistency** — do the same particles stay near each maximum?",
    "05_decoder_CC.png": "**05 · Map–map CC** between consecutive sampled epochs, per maximum. "
                         "High and flat = the decoder has converged.",
    "06_decoder_FSC.png": "**06 · Map–map FSC**, resolution-resolved. X-axis is 1/px.",
    "07_decoder_FSC-nyquist.png": "**07 · FSC at Nyquist** per epoch — the one-number summary "
                                  "of 06. A plateau is convergence.",
}
shown = []
for fname in sorted(CAPS):
    p = os.path.join(CDIR, "plots", fname)
    if os.path.exists(p):
        display(Markdown(CAPS[fname])); display(Image(p, width=width)); shown.append(fname)
print(f"{len(shown)} plot(s) from {os.path.join(CDIR, 'plots')}")
missing = [f for f in CAPS if f not in shown]
if missing:
    print("not produced:", missing)
    if any(f.startswith(("05", "06", "07")) for f in missing):
        print("  05-07 need mode='full' in cell 3.1 (and a GPU to be quick about it).")

## 5 · Save to Drive

The plots, the per-epoch UMAP pickles and `vector_metrics.pkl` are small. The generated volumes
(`vols.N/`) are not, and are rarely worth keeping — they exist to be compared with each other,
and the comparison is already in plots 05–07.

In [ ]:
#@title 5.1 · Copy results to Drive { display-mode: "form" }
#@markdown Blank = `<project>/convergence/<run>.<epoch>`.
dest_dir = ""  #@param {type:"string"}
#@markdown Also copy the generated + masked volumes (`vols.N/`). Usually unnecessary.
include_volumes = False  #@param {type:"boolean"}

import os, glob, shutil
CDIR = os.environ["CV_CDIR"]
DRIVE_DIR, NAME, E = os.environ["CV_DRIVE"], os.environ["CV_NAME"], os.environ["CV_E"]
DEST = os.path.abspath(dest_dir.strip()) if dest_dir.strip() \
       else os.path.join(DRIVE_DIR, "convergence", f"{NAME}.{E}")

todo, nbytes = [], 0
for root, _, files in os.walk(CDIR):
    for f in files:
        src = os.path.join(root, f)
        if os.path.islink(src):
            continue
        rel = os.path.relpath(src, CDIR)
        if rel.startswith("vols.") and not include_volumes:
            continue
        todo.append((src, os.path.join(DEST, rel)))
        nbytes += os.path.getsize(src)

print(f"copying {len(todo)} file(s), {nbytes/2**20:.1f} MiB → {DEST}")
for s, d in todo:
    os.makedirs(os.path.dirname(d), exist_ok=True)
    if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(s):
        shutil.copyfile(s, d)
nvol = len(glob.glob(os.path.join(CDIR, "vols.*", "*.mrc")))
if nvol and not include_volumes:
    print(f"  ({nvol} volume file(s) left behind — tick include_volumes to keep them)")
print(f"✅ {DEST}")
print("\nNothing was written to your model folder — the seeded z.0.pkl and everything else")
print(f"stayed in {os.environ['CV_WORK']}, which the runtime wipes when it recycles.")

---
### Notes

**The three bugs, precisely.**

1. *`z.0.pkl`.* `main()` asserts `z.{i}.pkl` exists for every `i` in `range(E)`
   (`analyze_convergence.py:1049-1053`), and `encoder_latent_shifts` hard-loads `z.0`, `z.1`
   and `z.2` (`:360-362`). But `train_vae` numbers epochs from 1 (`train_vae.py:909`), so
   `z.0.pkl` never exists. Cell 2.2 seeds it from `z.1.pkl` **inside the mirror**, which makes
   the first point of plot 02 a comparison of an epoch with itself — degenerate, so read that
   plot from epoch 3 onward.
2. *Volume indexing.* `mask_volumes` opens `vol_{cluster:03d}.mrc` for `cluster` in
   `range(len(labels))` — 0-based (`:830-833`) — while `analysis.gen_volumes` defaults to
   `vol_start_index=1`. Nothing writes `vol_000.mrc`, so masking fails and CC/FSC never run.
   Cell 3.1 wraps `gen_volumes` to force 0.
3. *The montage.* `encoder_latent_umaps` builds a `ceil(√N) × ceil(N/ceil(√N))` grid and then
   reads `epochs[i]` on the line *above* its own `try:` (`:306-307`), so its `except IndexError`
   is unreachable. Only N ∈ {1, 2, 4, 6, 9, 12, 16, 20, 25, 30, 36} exactly fill such a grid;
   every other count crashes on the spare axes. Cell 2.1 warns when your interval produces a bad
   count and suggests ones that do not; 3.1 catches the crash and redraws from the saved UMAPs,
   which are all written before it happens.

**Every epoch from 1 to E must be present.** Not just the sampled ones — the shift metric walks
consecutive epochs. Cell 2.1 checks and tells you the highest usable epoch if there is a gap.

**`--Apix` never affects the numbers here.** Unlike `analyze_landscape`, whose `--dilate` is in
Ångströms, this module's `--dilate` and `--dist` are in **voxels**, the CC is scale-free, and the
FSC x-axis is in 1/px. Å/px only reaches the MRC headers — which still matters if you open the
volumes in ChimeraX, hence cell 2.2 resolving it.

**Reading convergence honestly.** These are self-consistency measures: they show the model has
stopped changing, not that it is right. A run can converge beautifully onto a bad answer if the
poses are wrong or the stack is contaminated. Convergence is necessary, not sufficient.

**No held-out set exists.** `train_vae` trains on every particle, so none of this speaks to
overfitting. Two epochs agreeing tells you the optimisation settled, not that the model
generalises.

**Cost.** `fast` is dominated by UMAP — a few minutes per sampled epoch at 50k particles on CPU.
`full` adds `final_maxima × len(epochs)` volume decodes plus masking and FSCs. `reuse_umaps`
makes a second pass with different volume settings much cheaper, and it is on by default.